In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

In [ ]:
user_df = pd.read_csv("C:/Users/AANNYA/Desktop/DTI datasets/final_fintech_dataset_no_scores_no_recommendations (1).csv")

fd_df = pd.read_csv("C:/Users/AANNYA/Desktop/DTI datasets/india_property_fd_banks_2020_2026_dataset.csv")

property_df = pd.read_csv("C:/Users/AANNYA/Desktop/DTI datasets/india_required_states_property_dataset_2020_2026 (1).csv")

mf_df = pd.read_csv("C:/Users/AANNYA/Desktop/DTI datasets/mutual_funds_investment_dataset.csv")

print("All datasets loaded successfully")

In [ ]:
print(user_df.shape)
print(fd_df.shape)
print(property_df.shape)
print(mf_df.shape)

user_df.head()

In [ ]:
user_df["total_expenses"] = (
    user_df["rent"] +
    user_df["maintenance"] +
    user_df["emi"] +
    user_df["credit_card_payment"] +
    user_df["electricity"] +
    user_df["gas"] +
    user_df["internet"] +
    user_df["mobile"] +
    user_df["groceries"] +
    user_df["transportation"] +
    user_df["medical"] +
    user_df["household_items"]
)

In [ ]:
user_df["remaining_income"] = user_df["total_income"] - user_df["total_expenses"]

In [ ]:
user_df["luxury_budget"] = user_df["remaining_income"] * 0.4
user_df["investment_budget"] = user_df["remaining_income"] * 0.6

In [ ]:
def risk_profile(age):

    if age <= 25:
        return "High"

    elif age <= 35:
        return "Medium"

    else:
        return "Low"

In [ ]:
def recommend_sip(risk):

    if risk == "High":
        funds = mf_df.sort_values(
            by="Expected_Return_Percent",
            ascending=False
        ).head(3)

    elif risk == "Medium":
        funds = mf_df.sort_values(
            by="Expected_Return_Percent",
            ascending=False
        ).head(2)

    else:
        funds = mf_df.sort_values(
            by="Expected_Return_Percent",
            ascending=False
        ).head(1)

    return list(funds["Fund_Type"].unique())

In [ ]:
def recommend_fd():

    latest = fd_df[fd_df["year"] == 2026]

    rates = {
        "SBI": latest["state_bank_of_india_fd_percent"].mean(),
        "HDFC": latest["hdfc_bank_fd_percent"].mean(),
        "ICICI": latest["icici_bank_fd_percent"].mean(),
        "Axis": latest["axis_bank_fd_percent"].mean(),
        "PNB": latest["punjab_national_bank_fd_percent"].mean(),
        "BOB": latest["bank_of_baroda_fd_percent"].mean(),
        "Canara": latest["canara_bank_fd_percent"].mean(),
        "Union Bank": latest["union_bank_of_india_fd_percent"].mean(),
        "Indian Bank": latest["indian_bank_fd_percent"].mean(),
        "Kotak": latest["kotak_mahindra_bank_fd_percent"].mean()
    }

    best_bank = max(rates, key=rates.get)

    return best_bank

In [ ]:
def recommend_insurance(age):

    if age < 25:
        return "Basic Health Insurance"

    elif age < 35:
        return "Health + Term Insurance"

    else:
        return "Family Comprehensive Insurance"

In [ ]:
def financial_advisor(income, expenses, age):

    remaining = income - expenses

    luxury = remaining * 0.4
    investment = remaining * 0.6

    risk = risk_profile(age)

    sip = recommend_sip(risk)

    fd = recommend_fd()

    insurance = recommend_insurance(age)

    print("\n------ AI Financial Advisor ------")

    print("Income:", income)
    print("Expenses:", expenses)
    print("Remaining:", remaining)

    print("\nBudget Allocation")
    print("Luxury:", luxury)
    print("Investment:", investment)

    print("\nRecommended Investments")

    print("SIP Funds:", sip)

    print("Best FD Bank:", fd)

    print("Insurance:", insurance)

In [ ]:
financial_advisor(
    income = 70000,
    expenses = 30000,
    age = 24
)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
features = user_df[[
    "age",
    "total_income",
    "total_expenses",
    "remaining_income",
    "investment_budget"
]]

In [ ]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(features)

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)

user_df["user_cluster"] = kmeans.fit_predict(scaled_features)

In [ ]:
print(user_df["user_cluster"].value_counts())

In [ ]:
plt.figure()

sns.scatterplot(
    x=user_df["total_income"],
    y=user_df["total_expenses"],
    hue=user_df["user_cluster"]
)

plt.title("User Financial Behaviour Clusters")

plt.show()

In [ ]:
cluster_labels = {
    0: "High Spender",
    1: "Balanced Investor",
    2: "High Saver"
}

user_df["cluster_type"] = user_df["user_cluster"].map(cluster_labels)

In [ ]:
def investment_strategy(cluster):

    if cluster == 0:
        return "More FD and Insurance (Low Risk)"

    elif cluster == 1:
        return "Balanced SIP + FD"

    else:
        return "Aggressive SIP Investment"

In [ ]:
def financial_advisor(income, expenses, age):

    remaining = income - expenses

    luxury = remaining * 0.4
    investment = remaining * 0.6

    risk = risk_profile(age)

    sip = recommend_sip(risk)

    fd = recommend_fd()

    insurance = recommend_insurance(age)

    # Predict cluster
    data = [[age, income, expenses, remaining, investment]]
    scaled = scaler.transform(data)

    cluster = kmeans.predict(scaled)[0]

    strategy = investment_strategy(cluster)

    print("\n------ AI Financial Advisor ------")

    print("Income:", income)
    print("Expenses:", expenses)
    print("Remaining:", remaining)

    print("\nLuxury Budget:", luxury)
    print("Investment Budget:", investment)

    print("\nCluster Type:", cluster_labels[cluster])

    print("\nStrategy:", strategy)

    print("\nRecommended Investments")

    print("SIP:", sip)
    print("FD Bank:", fd)
    print("Insurance:", insurance)

In [ ]:
financial_advisor(
    income=7000000,
    expenses=30000,
    age=20
)

In [ ]:
def financial_health_score(income, expenses, investment):

    savings_ratio = (income - expenses) / income
    expense_ratio = expenses / income
    investment_ratio = investment / income

    score = 0

    # savings score
    if savings_ratio >= 0.4:
        score += 40
    elif savings_ratio >= 0.25:
        score += 30
    elif savings_ratio >= 0.1:
        score += 20
    else:
        score += 10

    # expense score
    if expense_ratio <= 0.5:
        score += 30
    elif expense_ratio <= 0.7:
        score += 20
    else:
        score += 10

    # investment score
    if investment_ratio >= 0.2:
        score += 30
    elif investment_ratio >= 0.1:
        score += 20
    else:
        score += 10

    return score

In [ ]:
def score_category(score):

    if score >= 80:
        return "Excellent Financial Health"

    elif score >= 60:
        return "Good Financial Health"

    elif score >= 40:
        return "Average Financial Health"

    else:
        return "Poor Financial Health"

In [ ]:
def financial_health_score(income, expenses, investment):

    savings_ratio = (income - expenses) / income
    expense_ratio = expenses / income
    investment_ratio = investment / income

    score = 0

    # Savings score (40 max)
    if savings_ratio >= 0.5:
        score += 35
    elif savings_ratio >= 0.35:
        score += 30
    elif savings_ratio >= 0.2:
        score += 20
    elif savings_ratio >= 0.1:
        score += 10
    else:
        score += 5

    # Expense score (30 max)
    if expense_ratio <= 0.4:
        score += 25
    elif expense_ratio <= 0.6:
        score += 20
    elif expense_ratio <= 0.8:
        score += 10
    else:
        score += 5

    # Investment score (30 max)
    if investment_ratio >= 0.3:
        score += 25
    elif investment_ratio >= 0.2:
        score += 20
    elif investment_ratio >= 0.1:
        score += 15
    else:
        score += 5

    return score

In [ ]:
financial_advisor(
    income=70000,
    expenses=3000,
    age=24
)

In [ ]:
def investment_allocation(investment_budget, risk):

    if risk == "High":

        sip = investment_budget * 0.60
        fd = investment_budget * 0.25
        insurance = investment_budget * 0.15

    elif risk == "Medium":

        sip = investment_budget * 0.45
        fd = investment_budget * 0.35
        insurance = investment_budget * 0.20

    else:

        sip = investment_budget * 0.30
        fd = investment_budget * 0.45
        insurance = investment_budget * 0.25

    return round(sip), round(fd), round(insurance)

In [ ]:
def financial_advisor(income, expenses, age):

    remaining = income - expenses

    luxury = remaining * 0.4
    investment = remaining * 0.6

    risk = risk_profile(age)

    sip = recommend_sip(risk)

    fd = recommend_fd()

    insurance = recommend_insurance(age)

    sip_amount, fd_amount, insurance_amount = investment_allocation(investment, risk)

    data = [[age, income, expenses, remaining, investment]]
    scaled = scaler.transform(data)

    cluster = kmeans.predict(scaled)[0]

    strategy = investment_strategy(cluster)

    score = financial_health_score(income, expenses, investment)

    category = score_category(score)

    print("\n------ AI Financial Advisor ------")

    print("Income:", income)
    print("Expenses:", expenses)
    print("Remaining:", remaining)

    print("\nLuxury Budget:", round(luxury))
    print("Investment Budget:", round(investment))

    print("\nCluster Type:", cluster_labels[cluster])

    print("\nStrategy:", strategy)

    print("\nFinancial Health Score:", score, "/100")
    print("Status:", category)

    print("\nRecommended Investments")

    print("SIP Funds:", sip)
    print("Best FD Bank:", fd)
    print("Insurance:", insurance)

    print("\nMonthly Investment Allocation")

    print("SIP Investment: ₹", sip_amount)
    print("FD Investment: ₹", fd_amount)
    print("Insurance: ₹", insurance_amount)

In [ ]:
financial_advisor(
    income=70000,
    expenses=30000,
    age=24
)